# Baseline: Qwen ASR
## Vietnamese-English Code-Switching ASR
Model: `Qwen/Qwen2-Audio-7B-Instruct`

> Qwen2-Audio is a multimodal model supporting multilingual ASR including Vietnamese. Uses instruction-following interface.

In [ ]:
!pip install -q \
    transformers==4.57.6 \
    accelerate==1.12.0 \
    evaluate==0.4.1 \
    jiwer==3.0.3 \
    librosa==0.10.1 \
    soundfile==0.12.1 \
    qwen-asr==0.0.6

In [ ]:
import os, re, json
import numpy as np
import pandas as pd
import torch
import librosa
from tqdm.notebook import tqdm
from transformers import AutoProcessor, Qwen2AudioForConditionalGeneration
import evaluate
from jiwer import wer

print(f"PyTorch: {torch.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

In [ ]:
# ── Config ─────────────────────────────────────────────────────────────────
MODEL_ID      = "Qwen/Qwen2-Audio-7B-Instruct"
AUDIO_DIR     = "./test/audio"
TRANSCRIPT    = "./test/transcript.json"
OUTPUT_CSV    = "./results_qwen_asr.csv"
SAMPLING_RATE = 16000
# Qwen2-Audio 7B nặng — chạy 1 file 1 lần để tránh OOM
BATCH_SIZE    = 1

os.environ["WANDB_DISABLED"] = "true"

In [ ]:
# ── Helpers ────────────────────────────────────────────────────────────────
def load_audio(path, target_sr=SAMPLING_RATE):
    audio, _ = librosa.load(path, sr=target_sr, mono=True)
    return audio[:30 * target_sr] if len(audio) > 30 * target_sr else audio

def normalize_text(text):
    text = text.lower()
    text = re.sub(r"[^\w\s]", "", text)
    return re.sub(r"\s+", " ", text).strip()

def compute_ser(refs, hyps):
    return sum(r.strip() != h.strip() for r, h in zip(refs, hyps)) / len(refs) * 100

def compute_all_metrics(refs, hyps):
    wer_m = evaluate.load("wer")
    cer_m = evaluate.load("cer")
    refs_n = [normalize_text(r) for r in refs]
    hyps_n = [normalize_text(h) for h in hyps]
    return {
        "wer_raw":  round(wer_m.compute(references=refs,   predictions=hyps)   * 100, 2),
        "cer_raw":  round(cer_m.compute(references=refs,   predictions=hyps)   * 100, 2),
        "ser_raw":  round(compute_ser(refs, hyps), 2),
        "wer_norm": round(wer_m.compute(references=refs_n, predictions=hyps_n) * 100, 2),
        "cer_norm": round(cer_m.compute(references=refs_n, predictions=hyps_n) * 100, 2),
        "ser_norm": round(compute_ser(refs_n, hyps_n), 2),
    }

def load_test_samples(audio_dir, transcript_path):
    with open(transcript_path, encoding="utf-8") as f:
        transcripts = json.load(f)
    samples, missing = [], 0
    for fname, meta in transcripts.items():
        path = os.path.join(audio_dir, fname)
        if os.path.exists(path):
            samples.append({"filename": fname, "audio_path": path, "text": meta["text"]})
        else:
            missing += 1
    print(f"Loaded {len(samples)} samples (skipped {missing} missing)")
    return samples

print("Helpers defined.")

In [ ]:
# ── Load model ─────────────────────────────────────────────────────────────
print(f"Loading {MODEL_ID}...")
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = Qwen2AudioForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
)
model.eval()
print("Model ready.")

In [ ]:
# ── Load test data ─────────────────────────────────────────────────────────
test_samples = load_test_samples(AUDIO_DIR, TRANSCRIPT)

In [ ]:
# ── Inference ──────────────────────────────────────────────────────────────
# Qwen2-Audio dùng chat template với audio token
PROMPT = "Please transcribe the following audio in its original language."

filenames, hypotheses, references = [], [], []

for s in tqdm(test_samples, desc="Inference"):
    audio = load_audio(s["audio_path"])

    conversation = [
        {
            "role": "user",
            "content": [
                {"type": "audio", "audio_url": s["audio_path"]},
                {"type": "text",  "text": PROMPT},
            ],
        }
    ]

    text_input = processor.apply_chat_template(
        conversation, add_generation_prompt=True, tokenize=False
    )
    inputs = processor(
        text=text_input,
        audios=[audio],
        sampling_rate=SAMPLING_RATE,
        return_tensors="pt",
    ).to("cuda")

    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=256)

    # Strip input tokens, decode only generated part
    generated = output_ids[:, inputs["input_ids"].shape[1]:]
    pred = processor.batch_decode(generated, skip_special_tokens=True)[0].strip()

    hypotheses.append(pred)
    filenames.append(s["filename"])
    references.append(s["text"])

print(f"Done — {len(hypotheses)} samples")

In [ ]:
# ── Metrics & save ─────────────────────────────────────────────────────────
metrics = compute_all_metrics(references, hypotheses)
print(f"Before norm → WER: {metrics['wer_raw']}% | CER: {metrics['cer_raw']}% | SER: {metrics['ser_raw']}%")
print(f"After  norm → WER: {metrics['wer_norm']}% | CER: {metrics['cer_norm']}% | SER: {metrics['ser_norm']}%")

df = pd.DataFrame({"filename": filenames, "reference": references, "hypothesis": hypotheses})
df["wer_per_sentence"] = df.apply(
    lambda r: wer(normalize_text(r["reference"]), normalize_text(r["hypothesis"])) * 100, axis=1
)
df.sort_values("wer_per_sentence", ascending=False).to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
print(f"Saved: {OUTPUT_CSV}")